# CDC Sync Status Check

This notebook verifies CDC replication between Aurora and TiDB databases.

**What it checks:**
- Table row counts in both databases
- CDC task status (running/paused/stopped)
- Replication lag (binlog position)
- Sample data comparison

In [1]:
# Import required libraries
import pymysql
import os
import json
import subprocess
from dotenv import load_dotenv
from datetime import datetime

## Load Environment Variables

In [2]:
# Load environment variables from .env file
load_dotenv()

# Aurora RDS MySQL Configuration
AURORA_HOST = os.getenv('AURORA_HOST')
AURORA_PORT = int(os.getenv('AURORA_PORT', 3306))
AURORA_USER = os.getenv('AURORA_USER', 'admin')
AURORA_PASSWORD = os.getenv('AURORA_PASSWORD')
AURORA_DATABASE = os.getenv('AURORA_DATABASE', 'ai_state_management')

# TiDB Configuration
TIDB_HOST = os.getenv('TIDB_HOST', '127.0.0.1')
TIDB_PORT = int(os.getenv('TIDB_PORT', 3306))
TIDB_USER = os.getenv('TIDB_USER', 'root')
TIDB_PASSWORD = os.getenv('TIDB_PASSWORD', '')
TIDB_DATABASE = os.getenv('TIDB_DATABASE', 'ai_state_management')

print("✓ Environment variables loaded")
print(f"  Aurora: {AURORA_HOST}")
print(f"  TiDB:   {TIDB_HOST}:{TIDB_PORT}")

✓ Environment variables loaded
  Aurora: database-2.c6loqao2038b.us-east-1.rds.amazonaws.com
  TiDB:   127.0.0.1:3306


---
## 1. CDC Task Status

Check if the CDC replication task is running.

In [ ]:
# Get CDC status from dmctl
try:
    result = subprocess.run(
        ['docker', 'exec', 'dm-master', '/dmctl', '--master-addr=dm-master:8261', 
         'query-status', 'aurora-to-tidb-cdc'],
        capture_output=True,
        text=True,
        timeout=10
    )
    
    if result.returncode == 0:
        cdc_status = json.loads(result.stdout)
        
        # Extract key information
        if cdc_status['result'] and cdc_status['sources']:
            source = cdc_status['sources'][0]
            subtask = source['subTaskStatus'][0]
            
            stage = subtask['stage']
            unit = subtask.get('unit', 'Unknown')
            
            print("CDC Replication Status")
            print("="*70)
            
            # Status
            status_emoji = "✓" if stage == "Running" else "✗"
            print(f"{status_emoji} Task Stage: {stage}")
            print(f"   Unit: {unit}")
            
            # Show appropriate info based on unit
            if unit == "Dump":
                print("\n📦 Currently dumping data from Aurora...")
                if 'dump' in subtask:
                    dump_info = subtask['dump']
                    print(f"   Status: {dump_info.get('dumpStatus', 'unknown')}")
                    
            elif unit == "Load":
                print("\n⬆️  Currently loading data into TiDB...")
                if 'load' in subtask:
                    load_info = subtask['load']
                    print(f"   Progress: {load_info.get('progress', 'unknown')}")
                    print(f"   Finished: {int(load_info.get('finishedBytes', 0)):,} bytes")
                    print(f"   Total: {int(load_info.get('totalBytes', 0)):,} bytes")
                    
            elif unit == "Sync":
                sync_info = subtask['sync']
                
                # Binlog positions
                print(f"\n📊 Binlog Positions:")
                print(f"   Aurora (master):  {sync_info['masterBinlog']}")
                print(f"   TiDB (syncer):    {sync_info['syncerBinlog']}")
                
                # Calculate lag
                master_pos = sync_info['masterBinlog']
                syncer_pos = sync_info['syncerBinlog']
                
                # Parse positions
                try:
                    import re
                    master_match = re.search(r'\.(\d+),\s*(\d+)', master_pos)
                    syncer_match = re.search(r'\.(\d+),\s*(\d+)', syncer_pos)
                    
                    if master_match and syncer_match:
                        master_file = int(master_match.group(1))
                        master_offset = int(master_match.group(2))
                        syncer_file = int(syncer_match.group(1))
                        syncer_offset = int(syncer_match.group(2))
                        
                        if master_file == syncer_file:
                            lag_bytes = master_offset - syncer_offset
                            lag_status = "✓" if lag_bytes < 1000 else "⚠"
                            print(f"\n{lag_status} Replication Lag: {lag_bytes:,} bytes")
                        else:
                            files_behind = master_file - syncer_file
                            print(f"\n⚠ Behind by {files_behind} binlog file(s)")
                except:
                    print(f"\n  (Could not parse lag information)")
                
                # Events and rows
                print(f"\n📈 Replication Stats:")
                print(f"   Total Events: {sync_info['totalEvents']}")
                print(f"   Total Rows:   {sync_info['totalRows']}")
                print(f"   Recent TPS:   {sync_info['recentTps']}")
            
            # Errors (check for all units)
            if subtask.get('result') and subtask['result'].get('errors'):
                print(f"\n✗ ERRORS DETECTED:")
                for err in subtask['result']['errors']:
                    print(f"  - {err['Message']}")
            else:
                print(f"\n✓ No errors")
            
            print("="*70)
        else:
            print("✗ CDC task not found or not configured")
    else:
        print(f"✗ Failed to get CDC status: {result.stderr}")
        
except subprocess.TimeoutExpired:
    print("✗ CDC status check timed out")
except Exception as e:
    print(f"✗ Error checking CDC status: {e}")

CDC Replication Status
✓ Task Stage: Running

Binlog Positions:
  Aurora (master):  (mysql-bin-changelog.000176, 157)
  TiDB (syncer):    (mysql-bin-changelog.000175, 1492)

⚠ Different binlog files (master: 176, syncer: 175)

Replication Stats:
  Total Events: 12
  Total Rows:   12
  Recent TPS:   0

✓ No errors


---
## 2. Connect to Databases

In [4]:
# Connect to Aurora
try:
    aurora_conn = pymysql.connect(
        host=AURORA_HOST,
        port=AURORA_PORT,
        user=AURORA_USER,
        password=AURORA_PASSWORD,
        database=AURORA_DATABASE,
        cursorclass=pymysql.cursors.DictCursor,
        connect_timeout=10
    )
    print(f"✓ Connected to Aurora ({AURORA_DATABASE})")
except Exception as e:
    print(f"✗ Aurora connection failed: {e}")
    raise

✓ Connected to Aurora (ai_state_management)


In [5]:
# Connect to TiDB
try:
    tidb_conn = pymysql.connect(
        host=TIDB_HOST,
        port=TIDB_PORT,
        user=TIDB_USER,
        password=TIDB_PASSWORD,
        database=TIDB_DATABASE,
        cursorclass=pymysql.cursors.DictCursor,
        connect_timeout=10
    )
    print(f"✓ Connected to TiDB ({TIDB_DATABASE})")
except Exception as e:
    print(f"✗ TiDB connection failed: {e}")
    raise

✓ Connected to TiDB (ai_state_management)


---
## 3. Compare Table Counts

Verify that both databases have the same number of records.

In [12]:
# Get table counts from both databases
tables = ['users', 'bots', 'sessions', 'messages', 'memory_snapshots']

aurora_counts = {}
tidb_counts = {}

print("Fetching table counts...\n")

# Aurora counts
with aurora_conn.cursor() as cursor:
    for table in tables:
        cursor.execute(f"SELECT COUNT(*) as count FROM {table}")
        aurora_counts[table] = cursor.fetchone()['count']

# TiDB counts
with tidb_conn.cursor() as cursor:
    for table in tables:
        cursor.execute(f"SELECT COUNT(*) as count FROM {table}")
        tidb_counts[table] = cursor.fetchone()['count']

# Display comparison
print("Table Count Comparison")
print("="*80)
print(f"{'Table':<25} {'Aurora':<20} {'TiDB':<20} {'Status':<15}")
print("="*80)

all_synced = True
total_aurora = 0
total_tidb = 0

for table in tables:
    aurora_count = aurora_counts[table]
    tidb_count = tidb_counts[table]
    
    total_aurora += aurora_count
    total_tidb += tidb_count
    
    if aurora_count == tidb_count:
        status = "✓ Synced"
    else:
        status = f"✗ Diff: {tidb_count - aurora_count:+,}"
        all_synced = False
    
    print(f"{table:<25} {aurora_count:<20,} {tidb_count:<20,} {status:<15}")

print("="*80)
print(f"{'TOTAL':<25} {total_aurora:<20,} {total_tidb:<20,} {'✓ Match' if all_synced else '✗ Mismatch':<15}")
print("="*80)

if all_synced:
    print("\n✓ All tables are synchronized!")
else:
    print("\n✗ Tables have different counts - CDC may be lagging or paused")

Fetching table counts...

Table Count Comparison
Table                     Aurora               TiDB                 Status         
users                     108                  103                  ✗ Diff: -5     
bots                      16                   16                   ✓ Synced       
sessions                  266                  263                  ✗ Diff: -3     
messages                  7,896                7,811                ✗ Diff: -85    
memory_snapshots          1,014                1,000                ✗ Diff: -14    
TOTAL                     9,300                9,193                ✗ Mismatch     

✗ Tables have different counts - CDC may be lagging or paused


---
## 4. Sample Data Comparison

Compare recent records to verify data quality.

In [ ]:
# Compare latest user records
print("Latest Users Comparison")
print("="*70)

with aurora_conn.cursor() as cursor:
    cursor.execute("SELECT user_id, username, email FROM users ORDER BY created_at DESC LIMIT 5")
    aurora_users = cursor.fetchall()

with tidb_conn.cursor() as cursor:
    cursor.execute("SELECT user_id, username, email FROM users ORDER BY created_at DESC LIMIT 5")
    tidb_users = cursor.fetchall()

print("\nAurora (latest 5):")
for user in aurora_users:
    print(f"  - {user['username']:<30} ({user['email']})")

print("\nTiDB (latest 5):")
for user in tidb_users:
    print(f"  - {user['username']:<30} ({user['email']})")

# Check if latest users match
if aurora_users and tidb_users:
    aurora_ids = set(u['user_id'] for u in aurora_users)
    tidb_ids = set(u['user_id'] for u in tidb_users)
    
    if aurora_ids == tidb_ids:
        print("\n✓ Latest users match between databases")
    else:
        print("\n✗ Latest users differ - checking sync lag...")
        missing_in_tidb = aurora_ids - tidb_ids
        if missing_in_tidb:
            print(f"  Missing in TiDB: {len(missing_in_tidb)} users")
else:
    print("\n⚠ No users found in one or both databases")

# Compare latest sessions (without ordering by timestamp)
print("\n" + "="*70)
print("Latest Sessions Comparison")
print("="*70)

with aurora_conn.cursor() as cursor:
    cursor.execute("SELECT session_id, user_id, bot_id FROM sessions LIMIT 5")
    aurora_sessions = cursor.fetchall()

with tidb_conn.cursor() as cursor:
    cursor.execute("SELECT session_id, user_id, bot_id FROM sessions LIMIT 5")
    tidb_sessions = cursor.fetchall()

print("\nAurora (sample 5 sessions):")
for sess in aurora_sessions:
    print(f"  - Session {sess['session_id'][:8]}... user={sess['user_id'][:8]}... bot={sess['bot_id'][:8]}...")

print("\nTiDB (sample 5 sessions):")
for sess in tidb_sessions:
    print(f"  - Session {sess['session_id'][:8]}... user={sess['user_id'][:8]}... bot={sess['bot_id'][:8]}...")

# Check if sample sessions overlap
if aurora_sessions and tidb_sessions:
    aurora_session_ids = set(s['session_id'] for s in aurora_sessions)
    tidb_session_ids = set(s['session_id'] for s in tidb_sessions)
    
    overlap = aurora_session_ids & tidb_session_ids
    if overlap:
        print(f"\n✓ {len(overlap)} sessions found in both databases")
    else:
        print("\n⚠ No overlap in sample sessions (different ordering)")
else:
    print("\n⚠ No sessions found in one or both databases")

Latest Users Comparison

Aurora (latest 5):
  - cdc.test.1775230089            (cdctest@example.com)
  - cdc.test.1775230000            (cdctest@example.com)
  - cdc.test.1775221607            (cdctest@example.com)
  - cdc.test.1775221277            (cdctest@example.com)
  - cdc.test.1775182509            (cdctest@example.com)

TiDB (latest 5):
  - cdc.test.1775182334            (cdctest@example.com)
  - cdc.test.1775181807            (cdctest@example.com)
  - cdc.test.1775181694            (cdctest@example.com)
  - bob.rodriguez755               (bob.rodriguez755@example.com)
  - tara.martinez332               (tara.martinez332@example.com)

✗ Latest users differ - checking sync lag...
  Missing in TiDB: 5 users


---
## 5. Recent Activity Check

Check for recent changes in the source database.

In [13]:
# Check recent activity in Aurora
print("Recent Activity (Last Hour)")
print("="*70)

with aurora_conn.cursor() as cursor:
    # Recent users
    cursor.execute("""
        SELECT COUNT(*) as count 
        FROM users 
        WHERE created_at >= NOW() - INTERVAL 1 HOUR
    """)
    recent_users = cursor.fetchone()['count']
    
    # Recent sessions (try 'created_at' first, then 'created')
    recent_sessions = 0
    try:
        cursor.execute("""
            SELECT COUNT(*) as count 
            FROM sessions 
            WHERE created_at >= NOW() - INTERVAL 1 HOUR
        """)
        recent_sessions = cursor.fetchone()['count']
    except:
        # If created_at doesn't exist, sessions table may not have timestamp column
        # Just count all sessions for comparison
        print("  (Note: sessions table missing timestamp column for recent activity check)")
    
    # Recent messages
    cursor.execute("""
        SELECT COUNT(*) as count 
        FROM messages 
        WHERE created_at >= NOW() - INTERVAL 1 HOUR
    """)
    recent_messages = cursor.fetchone()['count']

print(f"Aurora activity in last hour:")
print(f"  Users:    {recent_users:,}")
print(f"  Sessions: {recent_sessions:,}")
print(f"  Messages: {recent_messages:,}")

total_recent = recent_users + recent_sessions + recent_messages

if total_recent > 0:
    print(f"\n⚠ {total_recent:,} recent changes detected - verifying CDC is replicating...")
else:
    print(f"\n✓ No recent changes in last hour")

Recent Activity (Last Hour)
  (Note: sessions table missing timestamp column for recent activity check)
Aurora activity in last hour:
  Users:    2
  Sessions: 0
  Messages: 0

⚠ 2 recent changes detected - verifying CDC is replicating...


---
## Summary

Overall sync health check.

In [14]:
# Generate summary
print("\n" + "="*70)
print("SYNC STATUS SUMMARY")
print("="*70)

# Check CDC task
try:
    if 'stage' in locals() and stage == "Running":
        print("✓ CDC Task: Running")
    else:
        print("✗ CDC Task: Not running or error")
except:
    print("⚠ CDC Task: Status unknown")

# Check table counts
if 'all_synced' in locals():
    if all_synced:
        print("✓ Table Counts: All synchronized")
    else:
        print("✗ Table Counts: Mismatched")
else:
    print("⚠ Table Counts: Not checked")

# Overall status
print("\n" + "="*70)
if 'all_synced' in locals() and all_synced and 'stage' in locals() and stage == "Running":
    print("✓✓✓ SYNC STATUS: HEALTHY ✓✓✓")
else:
    print("⚠⚠⚠ SYNC STATUS: NEEDS ATTENTION ⚠⚠⚠")
print("="*70)


SYNC STATUS SUMMARY
✓ CDC Task: Running
✗ Table Counts: Mismatched

⚠⚠⚠ SYNC STATUS: NEEDS ATTENTION ⚠⚠⚠


---
## Close Connections

In [ ]:
# Close database connections
if 'aurora_conn' in locals() and aurora_conn.open:
    aurora_conn.close()
    print("✓ Aurora connection closed")

if 'tidb_conn' in locals() and tidb_conn.open:
    tidb_conn.close()
    print("✓ TiDB connection closed")